# The Derma Co – Extract All Available Product Reviews

This notebook is configured for the Sali-Cinamide product page you supplied. The product page currently displays **162 total reviews**. The earlier scraper stopped after the first small batch because its Load More detection was too narrow.

This version runs Playwright in a separate Python process (avoiding Jupyter's event-loop conflicts), detects the visible **Load More** button by role/text and common Judge.me selectors, and keeps clicking until no more reviews load or the page stops making progress.

Run all cells from top to bottom.

In [1]:
%pip install -q pandas playwright
!python -m playwright install chromium

Note: you may need to restart the kernel to use updated packages.


## 1. Create the scraper script

In [2]:
%%writefile scrape_all_reviews.py
import re
import time
import pandas as pd
from playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeoutError

PRODUCT_URL = (
    "https://thedermaco.com/products/"
    "2-sali-cinamide-anti-acne-face-wash-with-2-salicylic-acid-"
    "2-niacinamide-150-ml-pack-of-2"
)

OUTPUT_CSV = "thedermaco_all_reviews.csv"
MAX_LOAD_MORE_CLICKS = 200
WAIT_AFTER_CLICK_MS = 1800
MAX_NO_PROGRESS = 3


def extract_reviews(page):
    found = {}
    cards = page.locator(".jdgm-rev")

    for i in range(cards.count()):
        try:
            row = cards.nth(i).evaluate("""el => {
                const txt = s => {
                    const n = el.querySelector(s);
                    return n ? n.innerText.trim() : "";
                };
                const rating = el.querySelector(".jdgm-rev__rating");
                const date = el.querySelector(".jdgm-rev__timestamp");
                const body = el.querySelector(".jdgm-rev__body");

                return {
                    review_id: el.getAttribute("data-review-id") || "",
                    product_name: el.getAttribute("data-product-title") || "",
                    product_url: el.getAttribute("data-product-url") || "",
                    reviewer_name: txt(".jdgm-rev__author"),
                    rating: rating ? rating.getAttribute("data-score") || "" : "",
                    review_title: txt(".jdgm-rev__title"),
                    review_text: body ? body.innerText
                        .replace(/\\s*Read more\\s*/gi, "").trim() : "",
                    review_date: date
                        ? date.getAttribute("datetime") || date.innerText.trim() : "",
                    verified_buyer: el.getAttribute("data-verified-buyer") || "",
                    helpful_votes: el.getAttribute("data-thumb-up-count") || "",
                    not_helpful_votes: el.getAttribute("data-thumb-down-count") || ""
                };
            }""")

            rid = row["review_id"]
            key = rid if rid else (
                row["reviewer_name"], row["review_date"], row["review_text"]
            )
            found[key] = row
        except Exception as exc:
            print(f"Could not parse review {i}: {exc}")

    return found


def click_load_more(page):
    # Try known Judge.me selectors first.
    selectors = [
        ".jdgm-rev-widg__load-more",
        ".jdgm-all-reviews__load-more",
        ".jdgm-load-more",
        ".jdgm-rev-widg__load-more-btn",
        "button[class*='load-more']",
        "a[class*='load-more']"
    ]

    for selector in selectors:
        loc = page.locator(selector).last
        try:
            if loc.count() and loc.is_visible() and loc.is_enabled():
                loc.scroll_into_view_if_needed()
                loc.click(timeout=5000)
                return True, selector
        except Exception:
            pass

    # The live page exposes a button labelled "Load More".
    try:
        loc = page.get_by_role("button", name=re.compile(r"load\\s*more", re.I)).last
        if loc.count() and loc.is_visible() and loc.is_enabled():
            loc.scroll_into_view_if_needed()
            loc.click(timeout=5000)
            return True, "button role/name"
    except Exception:
        pass

    # Fallback for a div/span styled as a button.
    try:
        loc = page.get_by_text(re.compile(r"^\\s*load\\s*more\\s*$", re.I)).last
        if loc.count() and loc.is_visible():
            loc.scroll_into_view_if_needed()
            loc.click(timeout=5000)
            return True, "visible Load More text"
    except Exception:
        pass

    # Last fallback: click a visible DOM element whose text contains Load More.
    try:
        clicked = page.evaluate("""() => {
            const els = [...document.querySelectorAll('button, a, [role="button"], div, span')];
            const el = els.reverse().find(e =>
                e.innerText && /^\\s*load\\s*more\\s*$/i.test(e.innerText) &&
                e.getClientRects().length
            );
            if (!el) return false;
            el.click();
            return true;
        }""")
        if clicked:
            return True, "DOM text fallback"
    except Exception:
        pass

    return False, "No visible Load More control"


def main():
    reviews = {}

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page(
            viewport={"width": 1440, "height": 1100},
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            )
        )

        try:
            response = page.goto(
                PRODUCT_URL,
                wait_until="domcontentloaded",
                timeout=60000
            )
            print("Product page HTTP status:", response.status if response else "unknown")
            page.wait_for_timeout(5000)

            try:
                page.locator(".jdgm-rev").first.wait_for(
                    state="attached", timeout=30000
                )
            except PlaywrightTimeoutError:
                print("Judge.me cards not detected. Saving page diagnostics.")
                page.screenshot(path="scraper_debug.png", full_page=True)
                print("Page title:", page.title())
                print("Review card count:", page.locator(".jdgm-rev").count())
                return

            no_progress = 0

            for click_no in range(MAX_LOAD_MORE_CLICKS + 1):
                # Expand review bodies where possible
                expanders = page.locator(
                    ".jdgm-rev__body-read-more, "
                    ".jdgm-rev__read-more, "
                    ".jdgm-rev__body a[aria-label='Read more']"
                )
                for j in range(expanders.count()):
                    try:
                        e = expanders.nth(j)
                        if e.is_visible():
                            e.click(timeout=1000)
                    except Exception:
                        pass

                before = len(reviews)
                reviews.update(extract_reviews(page))
                after = len(reviews)
                print(
                    f"Pass {click_no + 1}: "
                    f"{page.locator('.jdgm-rev').count()} cards in DOM; "
                    f"{after} unique reviews collected."
                )

                if click_no == MAX_LOAD_MORE_CLICKS:
                    print("Reached MAX_LOAD_MORE_CLICKS safety limit.")
                    break

                clicked, method = click_load_more(page)
                if not clicked:
                    print(method)
                    break

                print("Clicked Load More using:", method)

                # Wait for newly appended/replaced review cards.
                try:
                    page.wait_for_function(
                        "(oldCount) => document.querySelectorAll('.jdgm-rev').length > oldCount",
                        arg=page.locator(".jdgm-rev").count(),
                        timeout=12000
                    )
                except PlaywrightTimeoutError:
                    page.wait_for_timeout(WAIT_AFTER_CLICK_MS)

                page.wait_for_timeout(500)
                reviews.update(extract_reviews(page))

                if len(reviews) <= before:
                    no_progress += 1
                    print(f"No new unique reviews after click ({no_progress}/{MAX_NO_PROGRESS}).")
                else:
                    no_progress = 0

                if no_progress >= MAX_NO_PROGRESS:
                    print("Stopping because repeated clicks did not add new reviews.")
                    break

            # Final extraction after pagination completes
            reviews.update(extract_reviews(page))

        finally:
            browser.close()

    columns = [
        "review_id", "product_name", "reviewer_name", "rating",
        "review_title", "review_text", "review_date", "verified_buyer",
        "helpful_votes", "not_helpful_votes", "product_url"
    ]

    df = pd.DataFrame(list(reviews.values()))
    for col in columns:
        if col not in df.columns:
            df[col] = ""
    df = df[columns]

    for col in ["rating", "helpful_votes", "not_helpful_votes"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print("\\n========== SCRAPING SUMMARY ==========")
    print("Unique reviews saved:", len(df))
    print("CSV file:", OUTPUT_CSV)
    print("Website displayed total: 162 (as shown when checked)")
    print("Compare the collected count with the website total.")
    if len(df) < 162:
        print(
            "WARNING: Fewer than 162 reviews were collected. "
            "The page may use another pagination/API mechanism; "
            "do not treat this partial CSV as the complete dataset."
        )


if __name__ == "__main__":
    main()


Overwriting scrape_all_reviews.py


## 2. Run the scraper

Run this cell. Keep the Jupyter kernel running while the separate Python process works.

In [3]:
!python scrape_all_reviews.py

^C


## 3. Load the exported CSV into Pandas

In [4]:
import pandas as pd

df = pd.read_csv("thedermaco_all_reviews.csv")

print("Dataset shape:", df.shape)
display(df.head(10))

Dataset shape: (162, 11)


,review_id,product_name,reviewer_name,rating,review_title,review_text,review_date,verified_buyer,helpful_votes,not_helpful_votes,product_url
0,e18ed183-d1fb-4f47-9e5c-84e2edb3010b,2% Sali-Cinamide Anti-Acne Face Wash with 2% S...,Tejas,5,Good,2% Sali-Cinamide Anti-Acne Face Wash with 2% S...,2026-09-13T05:10:00Z,False,0,0,/products/2-sali-cinamide-anti-acne-face-wash-...
1,8d555059-7343-4220-a9aa-98144cd5156e,2% Sali-Cinamide Anti-Acne Face Wash with 2% S...,Kabibunisa,5,NaN,It is the super product for pimple remove,2026-08-14T02:29:15Z,False,0,0,/products/2-sali-cinamide-anti-acne-face-wash-...
2,a4d3af1b-81db-4efd-8667-1dc49e7deb3d,2% Sali-Cinamide Anti-Acne Face Wash with 2% S...,Sayma,5,Fight acne and oil control,"Yeh derma ka best facewash h isme niacinamide,...",2026-08-07T05:40:16Z,False,0,0,/products/2-sali-cinamide-anti-acne-face-wash-...
3,1c619545-cd14-477e-9c28-5abe6fc21b9b,2% Sali-Cinamide Anti-Acne Face Wash with 2% S...,Athar .,5,NaN,2% Sali-Cinamide Anti-Acne Face Wash with 2% S...,2026-08-02T07:12:10Z,False,0,0,/products/2-sali-cinamide-anti-acne-face-wash-...
4,58d6f486-ec74-469e-b82b-ad8c4190fdfc,2% Sali-Cinamide Anti-Acne Face Wash with 2% S...,Lucky,5,NaN,Nice,2026-02-22T05:52:29Z,False,0,0,/products/2-sali-cinamide-anti-acne-face-wash-...
5,4a333700-364b-41d4-87f8-36bc38cb1181,2% Sali-Cinamide Anti-Acne Face Wash with 2% S...,pooja Yadav,5,NaN,2% Sali-Cinamide Anti-Acne Face Wash with 2% S...,2026-02-22T04:13:41Z,False,0,0,/products/2-sali-cinamide-anti-acne-face-wash-...
6,59551f8c-6aa4-4abb-892e-d793edfe2bdd,2% Sali-Cinamide Anti-Acne Face Wash with 2% S...,Chaiti,5,NaN,This product is super awesome. I started using...,2026-01-15T11:13:20Z,False,0,0,/products/2-sali-cinamide-anti-acne-face-wash-...
7,462ae070-f9ca-45b8-85ab-c6ec22afed17,2% Sali-Cinamide Anti-Acne Face Wash with 2% S...,PIYUSH KUMAR MEENA,5,Absolutely w...,Absolutely worth it product\nI will give 9 out...,2026-01-03T05:10:00Z,False,0,0,/products/2-sali-cinamide-anti-acne-face-wash-...
8,bd04ae5d-94e6-4400-b69d-0d0549309492,2% Sali-Cinamide Anti-Acne Face Wash with 2% S...,Shagun Agarwal,5,This anti ac...,This anti acne face wash is good for treating ...,2025-12-31T02:46:00Z,False,0,0,/products/2-sali-cinamide-anti-acne-face-wash-...
9,08fd6d6e-8568-48bc-a41f-505933ca988e,2% Sali-Cinamide Anti-Acne Face Wash with 2% S...,Rujal patel,5,This product...,This product is great for sensitive skin and i...,2025-12-30T15:46:00Z,False,0,0,/products/2-sali-cinamide-anti-acne-face-wash-...


## 4. Validate collection

In [7]:
print("Reviews collected:", len(df))
print("Unique review IDs:", df["review_id"].nunique())
print("Duplicate review IDs:", df["review_id"].duplicated().sum())

print("\\nRating distribution:")
display(df["rating"].value_counts(dropna=False).sort_index())

print("\\nMissing values:")
display(df.isna().sum().to_frame("missing_values"))

Reviews collected: 162
Unique review IDs: 162
Duplicate review IDs: 0
\nRating distribution:


rating
1      4
4     26
5    132
Name: count, dtype: int64

\nMissing values:


,missing_values
review_id,0
product_name,0
reviewer_name,0
rating,0
review_title,9
review_text,0
review_date,0
verified_buyer,0
helpful_votes,0
not_helpful_votes,0


## If it still collects fewer than 162

The page may use a custom endpoint or a pagination mechanism beyond the visible button. The script prints each pagination step and creates `scraper_debug.png` if it cannot detect the review widget. Share the full output from the scraper cell and, if created, the debug screenshot so the pagination can be adapted. Do not assume a partial CSV contains all reviews.

Use this scraper only for publicly available reviews and in accordance with the site's terms and access rules.

In [8]:
# Create a copy for preprocessing
df_clean = df.copy()

# Handle missing review titles
df_clean["review_title"] = (
    df_clean["review_title"].fillna("")
)

# Combine title and review text
df_clean["combined_review"] = (
    df_clean["review_title"] + " " +
    df_clean["review_text"]
).str.strip()

# Remove unnecessary whitespace
df_clean["combined_review"] = (
    df_clean["combined_review"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Check dataset
print("Original records:", len(df))
print("Cleaned records:", len(df_clean))
print("Missing combined reviews:",
      df_clean["combined_review"].isna().sum())

display(df_clean[[
    "rating",
    "review_title",
    "review_text",
    "combined_review"
]].head())

Original records: 162
Cleaned records: 162
Missing combined reviews: 0


,rating,review_title,review_text,combined_review
0,5,Good,2% Sali-Cinamide Anti-Acne Face Wash with 2% S...,Good 2% Sali-Cinamide Anti-Acne Face Wash with...
1,5,,It is the super product for pimple remove,It is the super product for pimple remove
2,5,Fight acne and oil control,"Yeh derma ka best facewash h isme niacinamide,...",Fight acne and oil control Yeh derma ka best f...
3,5,,2% Sali-Cinamide Anti-Acne Face Wash with 2% S...,2% Sali-Cinamide Anti-Acne Face Wash with 2% S...
4,5,,Nice,Nice


Product page HTTP status: 200
Pass 1: 3 cards in DOM; 3 unique reviews collected.
Clicked Load More using: button[class*='load-more']
Pass 2: 6 cards in DOM; 6 unique reviews collected.
Clicked Load More using: button[class*='load-more']
Pass 3: 9 cards in DOM; 9 unique reviews collected.
Clicked Load More using: button[class*='load-more']
Pass 4: 12 cards in DOM; 12 unique reviews collected.
Clicked Load More using: button[class*='load-more']
Pass 5: 15 cards in DOM; 15 unique reviews collected.
Clicked Load More using: button[class*='load-more']
Pass 6: 18 cards in DOM; 18 unique reviews collected.
Clicked Load More using: button[class*='load-more']
Pass 7: 21 cards in DOM; 21 unique reviews collected.
Clicked Load More using: button[class*='load-more']
Pass 8: 24 cards in DOM; 24 unique reviews collected.
Clicked Load More using: button[class*='load-more']
Pass 9: 27 cards in DOM; 27 unique reviews collected.
Clicked Load More using: button[class*='load-more']
Pass 10: 30 cards in D